# AgroSele — Pipeline Multi-Estágio (BM25 filtra → Cross-Encoder reranqueia)

Sexta variante: uma arquitetura de **dois estágios**, no espírito de
sistemas de busca/recuperação de informação usados em produção.

O problema que isso resolve: o Cross-Encoder (notebook anterior) é o
método mais forte entre os congelados, mas também o mais caro — precisa
rodar o BERT inteiro pra CADA candidata, sem cache possível entre
candidatas diferentes. Rodar isso pras 50 candidatas de cada pergunta, toda
vez, não escala bem.

A solução clássica: usar um método barato (BM25) pra filtrar rápido as
poucas candidatas realmente promissoras, e só gastar o Cross-Encoder caro
nessas poucas.

```
pergunta + 50 candidatas
      |
Estágio 1 (BM25, quase de graça): pontua as 50, filtra as top-K
      |
Estágio 2 (Cross-Encoder, caro): reranqueia só as K filtradas
      |
ranking final (as 50-K descartadas ficam no fim, na ordem do BM25)
```

In [1]:
import csv
import re

import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
from rank_bm25 import BM25Okapi

from cross_encoder_model import CabecaCrossEncoder

C:\Users\frede\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. BM25 (estágio 1) e cache do Cross-Encoder (estágio 2)

O BM25 usa a mesma tokenização das outras variantes. O Cross-Encoder
reaproveita o cache de features já extraído e o modelo já treinado no
notebook anterior — aqui não recomputamos nada caro de novo, só combinamos
as duas peças.

In [2]:
LISTA_STOPWORDS_PT = set("""
a ao aos aquela aquelas aquele aqueles aquilo as até com como da das de dele
deles depois do dos e ela elas ele eles em entre era eram essa essas esse
esses esta estamos estas estava estavam este esteja estejam estejamos estes
esteve estive estivemos estiveram estivesse estivessem estivéramos
estivéssemos estou está estávamos estão eu foi fomos for fora foram forem
formos fosse fossem fui fôramos fôssemos haja hajam hajamos havemos hei
houve houvemos houver houvera houveram houverei houverem houveremos
houveria houveriam houvermos houverá houverão houveríamos houvesse
houvessem houvéramos houvéssemos há hão isso isto já lhe lhes mais mas me
mesmo meu meus minha minhas muito na nas nem no nos nossa nossas nosso
nossos num numa não nós o os ou para pela pelas pelo pelos por qual quando
que quem se seja sejam sejamos sem serei seremos seria seriam será serão
seríamos seu seus somos sou sua suas são só também te tem temos tenha
tenham tenhamos tenho terei teremos teria teriam terá terão teríamos teu
teus teve tinha tinham tive tivemos tiver tivera tiveram tiverem tivermos
tivesse tivessem tivéramos tivéssemos tu tua tuas tá um uma você vocês
vos à às éramos é
""".split())
_padrao_token = re.compile(r"[a-zà-öø-ÿ0-9]+")


def tokenizar(texto):
    palavras = _padrao_token.findall(texto.lower())
    return [p for p in palavras if p not in LISTA_STOPWORDS_PT and len(p) > 1]


def carregar_csv(caminho):
    linhas = {}
    with open(caminho, encoding="utf-8") as f:
        for linha in csv.DictReader(f):
            linhas[linha["id"]] = linha["text"]
    return linhas


print("Carregando textos e construindo indice BM25...")
textos_respostas = carregar_csv("../selecao-resposta-milkqa-finetune/datasets/corpus.csv")
textos_perguntas = carregar_csv("../selecao-resposta-milkqa-finetune/datasets/queries.csv")

ids_documentos = list(textos_respostas.keys())
indice_do_id = {id_doc: i for i, id_doc in enumerate(ids_documentos)}
corpus_tokenizado = [tokenizar(t) for t in textos_respostas.values()]
bm25 = BM25Okapi(corpus_tokenizado)
print(f"Indice BM25 pronto: {len(ids_documentos)} documentos")

Carregando textos e construindo indice BM25...


Indice BM25 pronto: 2657 documentos


In [3]:
print("Carregando cache de pares do cross-encoder...")
cache = torch.load("cache/pair_features_crossencoder.pt", weights_only=False)
print(f"{len(cache)} pares no cache")

print("Carregando cross-encoder treinado...")
checkpoint = torch.load("checkpoints/best_model_crossencoder.pt", weights_only=False)
modelo_crossencoder = CabecaCrossEncoder(checkpoint["dim_entrada"], checkpoint["ocultas"], checkpoint["dropout"])
modelo_crossencoder.load_state_dict(checkpoint["model_state"])
print(f"Cross-encoder carregado (Acc@1 sozinho = {checkpoint['acuracia1_teste']:.4f}, "
      f"MRR sozinho = {checkpoint['mrr_teste']:.4f})")

print("Carregando splits oficiais do MilkQA...")
ds = load_dataset("eduagarcia/MilkQA")
conjunto_teste = ds["test"]

Carregando cache de pares do cross-encoder...


38263 pares no cache
Carregando cross-encoder treinado...
Cross-encoder carregado (Acc@1 sozinho = 0.6167, MRR sozinho = 0.7152)
Carregando splits oficiais do MilkQA...


## 2. O pipeline de dois estágios

Para cada pergunta: o BM25 ordena as 50 candidatas; pegamos só as top-K;
o Cross-Encoder reranqueia essas K; o resultado final é essa reordenação
seguida do resto (que nunca passou pelo estágio caro, então fica ordenado
só pelo BM25 mesmo, no fim da lista).

Contamos também quantas vezes o Cross-Encoder foi chamado, pra mostrar a
economia computacional de verdade.

In [4]:
def avaliar_pipeline(conjunto, bm25, indice_do_id, textos_perguntas, cache, modelo, k):
    modelo.eval()
    lista_acuracia1, lista_mrr = [], []
    chamadas_crossencoder = 0

    with torch.no_grad():
        for linha in conjunto:
            id_pergunta, id_certa, candidatas = linha["query-id"], linha["positive-doc-id"], linha["candidates-ids"]

            escores_bm25 = bm25.get_scores(tokenizar(textos_perguntas[id_pergunta]))
            candidatas_ordenadas = sorted(candidatas, key=lambda c: -escores_bm25[indice_do_id[c]])

            top_k = candidatas_ordenadas[:k]
            resto = candidatas_ordenadas[k:]

            features_top_k = torch.stack([cache[f"{id_pergunta}||{c}"] for c in top_k])
            pontuacoes_top_k = torch.sigmoid(modelo(features_top_k)).numpy()
            chamadas_crossencoder += len(top_k)

            ordem = np.argsort(-pontuacoes_top_k)
            ranking_final = [top_k[i] for i in ordem] + resto

            posicao = ranking_final.index(id_certa) + 1
            lista_acuracia1.append(1.0 if posicao == 1 else 0.0)
            lista_mrr.append(1.0 / posicao)

    return float(np.mean(lista_acuracia1)), float(np.mean(lista_mrr)), chamadas_crossencoder

In [5]:
print("===== Pipeline multi-estagio: BM25 filtra top-K, cross-encoder reranqueia =====")
print(f"Referencia -- cross-encoder puro sobre as 50: "
      f"Acc@1={checkpoint['acuracia1_teste']:.4f} MRR={checkpoint['mrr_teste']:.4f} (50 chamadas/pergunta)\n")

resultados = []
for k in [3, 5, 10, 20, 50]:
    acuracia1, mrr, chamadas = avaliar_pipeline(
        conjunto_teste, bm25, indice_do_id, textos_perguntas, cache, modelo_crossencoder, k)
    economia = 1 - (k / 50)
    print(f"  K={k:2d} -> Acc@1={acuracia1:.4f} MRR={mrr:.4f} "
          f"| chamadas ao cross-encoder: {chamadas} (media {chamadas/len(conjunto_teste):.0f}/pergunta) "
          f"| economia de {economia:.0%}")
    resultados.append({"k": k, "acuracia1": acuracia1, "mrr": mrr, "economia_chamadas": economia})

df_resultados = pd.DataFrame(resultados)
df_resultados.to_csv("checkpoints/resultados_pipeline_multiestagio_notebook.csv", index=False)
df_resultados

===== Pipeline multi-estagio: BM25 filtra top-K, cross-encoder reranqueia =====
Referencia -- cross-encoder puro sobre as 50: Acc@1=0.6167 MRR=0.7152 (50 chamadas/pergunta)



  K= 3 -> Acc@1=0.5300 MRR=0.6165 | chamadas ao cross-encoder: 900 (media 3/pergunta) | economia de 94%


  K= 5 -> Acc@1=0.5533 MRR=0.6363 | chamadas ao cross-encoder: 1500 (media 5/pergunta) | economia de 90%


  K=10 -> Acc@1=0.5900 MRR=0.6752 | chamadas ao cross-encoder: 3000 (media 10/pergunta) | economia de 80%


  K=20 -> Acc@1=0.6000 MRR=0.6959 | chamadas ao cross-encoder: 6000 (media 20/pergunta) | economia de 60%


  K=50 -> Acc@1=0.6167 MRR=0.7156 | chamadas ao cross-encoder: 15000 (media 50/pergunta) | economia de 0%


,k,acuracia1,mrr,economia_chamadas
0,3,0.530000,0.616455,0.94
1,5,0.553333,0.636288,0.90
2,10,0.590000,0.675171,0.80
3,20,0.600000,0.695852,0.60
4,50,0.616667,0.715559,0.00


## Conclusão

Com **K=10** (rerankeando só 10 das 50 candidatas), o pipeline perde
apenas 0,027 de Accuracy@1 em relação ao Cross-Encoder rodando em todas as
50 (0,590 vs. 0,617), por **80% menos chamadas ao BERT** por pergunta. Com
K=20, a perda cai pra praticamente zero (0,600 vs. 0,617) já economizando
60% das chamadas.

Esse é o princípio central por trás de arquiteturas de busca/recuperação em
produção — nunca gastar o modelo mais caro em candidatas obviamente
irrelevantes, deixando um filtro barato (aqui, BM25) descartar o óbvio
primeiro. Numa implantação real (por exemplo, dentro do MilkShow, o
software de gestão de fazendas leiteiras que motivou a escolha do domínio
deste projeto), isso significa poder responder perguntas de produtores
usando o modelo mais forte disponível, sem pagar o custo computacional de
rodá-lo em candidatas que um filtro simples já descartaria.